# Python Fundamentals for Data Analytics: World Happiness Report 2019

This notebook applies **pandas, NumPy, and Matplotlib** to a real-world dataset from the **World Happiness Report 2019**. It covers exploratory data analysis (EDA), data cleaning, filtering, grouping, merging, reshaping, reusable functions, loops, conditionals, and two Matplotlib visualizations.

**Dataset:** 156 countries/regions with happiness scores and explanatory factors including GDP per capita, social support, healthy life expectancy, freedom, generosity, and perceptions of corruption.

> In Google Colab, choose **Runtime → Run all** before submission so all outputs and figures are saved in the notebook.

In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

## 1. Load and inspect the dataset

In [ ]:
# Public CSV mirror of the World Happiness Report 2019 dataset
url = 'https://gist.githubusercontent.com/mshoer/92aed7f3ad5efc3519088b27724d07ce/raw/3022ba8848946489e3a0670e64e95834437b7a6e/2019.csv'

df = pd.read_csv(url)
print('Dataset shape:', df.shape)
display(df.head())

In [ ]:
# Inspect structure, data types, and missing values
print('Column information:')
df.info()

print('\nMissing values by column:')
display(df.isna().sum().to_frame('missing_values'))

print('\nDescriptive statistics:')
display(df.describe())

## 2. Clean and prepare the data

In [ ]:
# Rename columns to concise snake_case names
rename_map = {
    'Overall rank': 'rank',
    'Country or region': 'country',
    'Score': 'score',
    'GDP per capita': 'gdp_per_capita',
    'Social support': 'social_support',
    'Healthy life expectancy': 'healthy_life_expectancy',
    'Freedom to make life choices': 'freedom',
    'Generosity': 'generosity',
    'Perceptions of corruption': 'corruption_perception'
}

df = df.rename(columns=rename_map).copy()

# Remove exact duplicate rows if any are present
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f'Removed {before - len(df)} duplicate rows.')
print('Cleaned shape:', df.shape)
display(df.head())

## 3. Custom function, conditionals, and filtering

In [ ]:
# Use a custom function with conditionals to classify happiness level
def classify_happiness(score):
    if score >= 6.0:
        return 'High'
    elif score >= 4.5:
        return 'Medium'
    else:
        return 'Low'

df['happiness_level'] = df['score'].apply(classify_happiness)

display(df[['country', 'score', 'happiness_level']].head(10))
print('\nCountries per happiness level:')
print(df['happiness_level'].value_counts())

In [ ]:
# Filter: examine the ten highest-ranked countries
top_10 = df[df['rank'] <= 10][['rank', 'country', 'score', 'gdp_per_capita', 'social_support']]
display(top_10)

## 4. Group and aggregate

In [ ]:
# Group countries by happiness level and compare average characteristics
group_summary = (
    df.groupby('happiness_level', observed=True)
      .agg(
          countries=('country', 'count'),
          avg_score=('score', 'mean'),
          avg_gdp=('gdp_per_capita', 'mean'),
          avg_social_support=('social_support', 'mean'),
          avg_health=('healthy_life_expectancy', 'mean'),
          avg_freedom=('freedom', 'mean')
      )
      .reindex(['High', 'Medium', 'Low'])
      .round(3)
)
display(group_summary)

## 5. Merge tables

In [ ]:
# Split the dataset into two logical tables, then merge them on country.
# This simulates combining measures from separate data sources.
score_table = df[['country', 'rank', 'score', 'happiness_level']].copy()
factor_table = df[['country', 'gdp_per_capita', 'social_support',
                   'healthy_life_expectancy', 'freedom',
                   'generosity', 'corruption_perception']].copy()

merged_df = pd.merge(score_table, factor_table, on='country', how='inner')
print('Merged shape:', merged_df.shape)
display(merged_df.head())

## 6. Reshape with melt and pivot_table

In [ ]:
factor_columns = [
    'gdp_per_capita', 'social_support', 'healthy_life_expectancy',
    'freedom', 'generosity', 'corruption_perception'
]

# Wide -> long format
long_df = merged_df.melt(
    id_vars=['country', 'happiness_level'],
    value_vars=factor_columns,
    var_name='factor',
    value_name='value'
)
print('Long-format shape:', long_df.shape)
display(long_df.head(12))

# Long -> summarized pivot table
factor_pivot = pd.pivot_table(
    long_df,
    values='value',
    index='happiness_level',
    columns='factor',
    aggfunc='mean'
).reindex(['High', 'Medium', 'Low']).round(3)

display(factor_pivot)

## 7. NumPy, loops, and correlations

In [ ]:
# Use NumPy to standardize the happiness score
score_array = df['score'].to_numpy()
df['score_z'] = (score_array - np.mean(score_array)) / np.std(score_array)

display(df[['country', 'score', 'score_z']].head())

In [ ]:
# Use a loop to calculate the correlation between each factor and happiness score
correlations = {}
for factor in factor_columns:
    corr = np.corrcoef(df[factor].to_numpy(), df['score'].to_numpy())[0, 1]
    correlations[factor] = corr
    print(f'{factor:25s}: {corr:.3f}')

strongest_factor = max(correlations, key=lambda x: abs(correlations[x]))
print(f'\nStrongest absolute correlation with happiness score: {strongest_factor} '
      f'({correlations[strongest_factor]:.3f})')

## 8. Visualization 1: Distribution of happiness scores

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df['score'], bins=12, edgecolor='black')
plt.axvline(df['score'].mean(), linestyle='--', label=f"Mean = {df['score'].mean():.2f}")
plt.title('Distribution of World Happiness Scores (2019)')
plt.xlabel('Happiness Score')
plt.ylabel('Number of Countries/Regions')
plt.legend()
plt.tight_layout()
plt.show()

## 9. Visualization 2: GDP per capita vs. happiness score

In [ ]:
x = df['gdp_per_capita'].to_numpy()
y = df['score'].to_numpy()

# NumPy linear trend line
slope, intercept = np.polyfit(x, y, 1)
x_line = np.linspace(x.min(), x.max(), 100)
y_line = slope * x_line + intercept

plt.figure(figsize=(8, 5))
plt.scatter(x, y, alpha=0.7)
plt.plot(x_line, y_line, linewidth=2, label='Linear trend')
plt.title('GDP per Capita and Happiness Score (2019)')
plt.xlabel('GDP per Capita Index')
plt.ylabel('Happiness Score')
plt.legend()
plt.tight_layout()
plt.show()

## 10. Key observations generated from the analysis

In [ ]:
print('Top-ranked country:')
display(df.nsmallest(1, 'rank')[['rank', 'country', 'score']])

print('Lowest-ranked country:')
display(df.nlargest(1, 'rank')[['rank', 'country', 'score']])

print(f"Average happiness score: {df['score'].mean():.3f}")
print(f"Median happiness score: {df['score'].median():.3f}")
print(f"GDP-score correlation: {correlations['gdp_per_capita']:.3f}")
print(f"Social-support-score correlation: {correlations['social_support']:.3f}")

## Conclusion

This analysis demonstrates a complete introductory data-analytics workflow. The dataset was loaded and inspected with pandas, cleaned and transformed, filtered and grouped, split and merged, reshaped with `melt` and `pivot_table`, processed with NumPy, automated with a function and loops, and visualized with Matplotlib. The results also illustrate that national happiness is associated with multiple socioeconomic and social-support factors rather than a single variable.